In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import os

In [ ]:
drive.mount('/content/drive')

# directories
data_folder = "/content/drive/My Drive/0. Liquidity and Market Stress/Python Code/batch_process_minute_e_spread/5-year_minute_data_by_coins/BTC"
event_file = "/content/drive/My Drive/0. Liquidity and Market Stress/Python Code/batch_process_minute_e_spread/panel_generator/event_filter(incl_ob_hour_&_types).csv"

# load data
event_data = pd.read_csv(event_file, parse_dates=["event_ob_hour"])

# initilize the 6 panels
metrics = ["spread", "depth", "volume", "avg_trade_price", "avg_e_spread", "n_trades"]
total_minutes = 1440 * 2 + 1  # pre-event 12 hours + event hour + post even 12 hours
time_index = list(range(-1440, 1441))  # initialize the first column
panel_dict = {metric: pd.DataFrame({"time": time_index}) for metric in metrics}  # get the time for the first column of each panel

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def generate_panels_for_coin(coin, input_folder, output_folder, event_file):
    """
    Generate Panel files for a specific cryptocurrency, including filling missing data for events on the 1st of each month.

    Parameters:
    coin (str): Cryptocurrency name (e.g., "BTC", "DOGE", "USDT", "SHIB")
    input_folder (str): Path to the folder containing single-month data files
    output_folder (str): Path to save the generated Panel files
    event_file (str): Path to the event file
    """
    # Load the event data
    event_data = pd.read_csv(event_file, parse_dates=["event_ob_hour"])

    # Initialize 6 Panels
    metrics = ["spread", "depth", "volume", "avg_trade_price", "avg_e_spread", "n_trades"]
    total_minutes = 1440 * 2 + 1  # 1440 minutes before + event time + 1440 minutes after
    time_index = list(range(-1440, 1441))  # Initialize the time column
    panel_dict = {metric: pd.DataFrame({"time": time_index}) for metric in metrics}  # Each Panel's first column is 'time'

    # Iterate through events
    for event_idx, event_row in event_data.iterrows():
        event_time = event_row["event_ob_hour"]
        event_year_month = event_time.strftime("%Y_%m")  # Get the year and month of the event
        event_minute = event_time.strftime("%H:%M")  # Get the hour and minute of the event

        # Locate the corresponding single-month CSV file
        monthly_file = f"minute_liquidity_{event_year_month}.csv"
        monthly_path = os.path.join(input_folder, monthly_file)

        if not os.path.exists(monthly_path):
            print(f"File not found: {monthly_path}, skipping event {event_idx + 1}")
            continue

        # Load the single-month data
        monthly_data = pd.read_csv(monthly_path, parse_dates=["datetime"])
        monthly_data.set_index("datetime", inplace=True)

        # Locate the row corresponding to the event time
        if event_time not in monthly_data.index:
            print(f"Event time {event_time} not found in {monthly_file}, skipping event {event_idx + 1}")
            continue

        event_row_idx = monthly_data.index.get_loc(event_time)

        # Extract data for 1440 minutes before and after the event
        start_idx = max(0, event_row_idx - 1440)  # Ensure the index does not go out of bounds
        end_idx = min(len(monthly_data), event_row_idx + 1440 + 1)
        event_window_data = monthly_data.iloc[start_idx:end_idx]

        # Fill data into each Panel
        for metric in metrics:
            panel = panel_dict[metric]
            metric_data = event_window_data[metric].values

            # If data is insufficient, pad with NaNs
            padded_data = np.full(total_minutes, np.nan)  # Initialize with NaNs
            offset = 1440 - (event_row_idx - start_idx)  # Calculate the offset
            padded_data[offset:offset + len(metric_data)] = metric_data

            # Add the padded data to the corresponding column in the Panel
            panel[f"event{event_idx + 1}"] = padded_data

    # Fill missing data for events on the 1st of each month
    for event_idx, event_row in event_data.iterrows():
        event_time = event_row["event_ob_hour"]

        # Check if the event is on the 1st of the month
        if event_time.day != 1:
            continue

        # Get the file name for the previous month
        previous_month = (event_time - pd.Timedelta(days=1)).strftime("%Y_%m")
        previous_file = f"minute_liquidity_{previous_month}.csv"
        previous_path = os.path.join(input_folder, previous_file)

        # Check if the file for the previous month exists
        if not os.path.exists(previous_path):
            print(f"Previous month file not found: {previous_path}, skipping missing data for event {event_idx + 1}")
            continue

        # Load the data for the previous month
        previous_data = pd.read_csv(previous_path, parse_dates=["datetime"])
        previous_data.set_index("datetime", inplace=True)

        # Calculate the number of missing minutes
        missing_minutes = (1440 - event_time.hour * 60 - event_time.minute)

        # Extract the last n rows from the previous month's data
        previous_tail_data = previous_data.iloc[-missing_minutes:]

        # Fill the extracted data into the corresponding Panel
        for metric in metrics:
            panel = panel_dict[metric]
            event_column = f"event{event_idx + 1}"

            # If the event column exists, fill the missing data
            if event_column in panel.columns:
                panel.loc[:missing_minutes - 1, event_column] = previous_tail_data[metric].values

    # Save the Panels to CSV files
    os.makedirs(output_folder, exist_ok=True)

    for metric, panel in panel_dict.items():
        output_file = os.path.join(output_folder, f"panel_{coin}_{metric}.csv")
        panel.to_csv(output_file, index=False)
        print(f"Saved {metric} panel for {coin} to {output_file}")

In [ ]:
# Main program: Generate Panel files for multiple cryptocurrencies
coins = ["BTC", "DOGE", "USDT", "SHIB"]
base_input_folder = "/content/drive/My Drive/0. Liquidity and Market Stress/Python Code/batch_process_minute_e_spread/5-year_minute_data_by_coins"
base_output_folder = "/content/drive/My Drive/0. Liquidity and Market Stress/Python Code/batch_process_minute_e_spread/panel_generator"
event_file = "/content/drive/My Drive/0. Liquidity and Market Stress/Python Code/batch_process_minute_e_spread/panel_generator/event_filter(incl_ob_hour_&_types).csv"

for coin in coins:
    input_folder = os.path.join(base_input_folder, coin)
    output_folder = os.path.join(base_output_folder, f"{coin}_panels")
    generate_panels_for_coin(coin, input_folder, output_folder, event_file)

Saved spread panel for BTC to /content/drive/My Drive/0. Liquidity and Market Stress/Python Code/batch_process_minute_e_spread/panel_generator/BTC_panels/panel_BTC_spread.csv
Saved depth panel for BTC to /content/drive/My Drive/0. Liquidity and Market Stress/Python Code/batch_process_minute_e_spread/panel_generator/BTC_panels/panel_BTC_depth.csv
Saved volume panel for BTC to /content/drive/My Drive/0. Liquidity and Market Stress/Python Code/batch_process_minute_e_spread/panel_generator/BTC_panels/panel_BTC_volume.csv
Saved avg_trade_price panel for BTC to /content/drive/My Drive/0. Liquidity and Market Stress/Python Code/batch_process_minute_e_spread/panel_generator/BTC_panels/panel_BTC_avg_trade_price.csv
Saved avg_e_spread panel for BTC to /content/drive/My Drive/0. Liquidity and Market Stress/Python Code/batch_process_minute_e_spread/panel_generator/BTC_panels/panel_BTC_avg_e_spread.csv
Saved n_trades panel for BTC to /content/drive/My Drive/0. Liquidity and Market Stress/Python Cod